# Byte Pair Encoding (BPE) From Scratch

Byte Pair Encoding (BPE) is a subword tokenization method that strikes a balance between character-level and word-level tokenization. It is widely used in modern LLMs like GPT-2, GPT-3, and Llama.

### Why BPE?
1. **Handles Out-of-Vocabulary (OOV) words**: By breaking words into subword units, it can represent unknown words as a sequence of known subwords.
2. **Efficiency**: Frequently occurring sequences of characters are merged into single tokens, reducing the sequence length compared to character-level tokenization.
3. **Fixed Vocabulary Size**: You can pre-define the vocabulary size, and the algorithm will find the most optimal units.

---

## 1. Initial Setup

We start with a simple corpus. In BPE, we typically treat each word as a sequence of characters, often adding a special suffix (like `</w>`) to denote the end of a word, which helps in reconstruction.

In [ ]:
import re
from collections import defaultdict

# Load sample text from file
try:
    with open("sample_text.txt", "r", encoding="utf-8") as f:
        text = f.read()
except FileNotFoundError:
    # Fallback if file is not found
    text = """Hello, world. Is this-- a test?
Tokenization is the first step in text processing.
It splits words, punctuation, and special symbols."""

# Pre-processing: simple word splitting and adding end-of-word marker
def initialize_vocab(text):
    vocab = defaultdict(int)
    words = text.split()
    for word in words:
        # Represent word as characters separated by spaces, ending with </w>
        char_word = " ".join(list(word)) + " </w>"
        vocab[char_word] += 1
    return vocab

vocab = initialize_vocab(text)
print("Initial Vocabulary (partial):")
for i, (word, freq) in enumerate(vocab.items()):
    print(f"{word}: {freq}")
    if i > 5: break

## 2. Counting Pairs

We need a function to find the most frequent pair of adjacent symbols in our current vocabulary.

In [ ]:
def get_stats(vocab):
    pairs = defaultdict(int)
    for word, frequency in vocab.items():
        symbols = word.split()
        for i in range(len(symbols)-1):
            pairs[symbols[i], symbols[i+1]] += frequency
    return pairs

pairs = get_stats(vocab)
best_pair = max(pairs, key=pairs.get)
print(f"Most frequent pair: {best_pair} with frequency {pairs[best_pair]}")

## 3. Merging Symbols

Once we find the most frequent pair, we merge all occurrences of that pair into a single symbol.

In [ ]:
def merge_vocab(pair, v_in):
    v_out = {}
    # Escape special characters in the pair for regex
    bigram = re.escape(' '.join(pair))
    p = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
    
    for word in v_in:
        # Replace the pair with the merged version
        w_out = p.sub(''.join(pair), word)
        v_out[w_out] = v_in[word]
    return v_out

new_vocab = merge_vocab(best_pair, vocab)
print(f"Vocab after merging {best_pair}:")
for i, (word, freq) in enumerate(new_vocab.items()):
    if ''.join(best_pair) in word:
        print(f"{word}: {freq}")
        break

## 4. The BPE Training Loop

We repeat the process for a fixed number of iterations to build our vocabulary.

In [ ]:
num_merges = 50
vocab = initialize_vocab(text)
merges = {} # Store learned merges to use for encoding

for i in range(num_merges):
    pairs = get_stats(vocab)
    if not pairs:
        break
    best = max(pairs, key=pairs.get)
    vocab = merge_vocab(best, vocab)
    merges[best] = ''.join(best)
    if i % 10 == 0:
        print(f"Iteration {i}: Merged {best}")

print("\nLearned Merges:", list(merges.values())[:10])

## 5. Implementing the BPETokenizer Class

Now we wrap everything into a clean class that can `encode` and `decode` text.

In [ ]:
class BPETokenizer:
    def __init__(self, num_merges=100):
        self.num_merges = num_merges
        self.merges = {}
        self.vocab = set()
        self.special_tokens = ["</w>"]

    def train(self, text):
        vocab = initialize_vocab(text)
        for i in range(self.num_merges):
            pairs = get_stats(vocab)
            if not pairs:
                break
            best = max(pairs, key=pairs.get)
            vocab = merge_vocab(best, vocab)
            self.merges[best] = ''.join(best)
        
        # Build final vocabulary from all seen tokens
        self.vocab = set(self.special_tokens)
        for word in vocab.keys():
            for token in word.split():
                self.vocab.add(token)
        
        # Create mappings
        self.token_to_id = {token: i for i, token in enumerate(sorted(list(self.vocab)))}
        self.id_to_token = {i: token for token, i in self.token_to_id.items()}

    def encode(self, text):
        words = text.split()
        encoded_ids = []
        
        for word in words:
            # Start with characters
            word_tokens = list(word) + ["</w>"]
            
            # Apply merges in the order they were learned
            while True:
                pairs = []
                for i in range(len(word_tokens)-1):
                    pairs.append((word_tokens[i], word_tokens[i+1]))
                
                # Find if any of our learned merges apply
                apply_merge = None
                for learned_pair in self.merges:
                    if learned_pair in pairs:
                        apply_merge = learned_pair
                        break # Apply the earliest learned merge found
                
                if not apply_merge:
                    break
                
                # Perform the merge
                new_tokens = []
                i = 0
                while i < len(word_tokens):
                    if i < len(word_tokens)-1 and (word_tokens[i], word_tokens[i+1]) == apply_merge:
                        new_tokens.append(''.join(apply_merge))
                        i += 2
                    else:
                        new_tokens.append(word_tokens[i])
                        i += 1
                word_tokens = new_tokens
            
            # Convert to IDs
            for token in word_tokens:
                encoded_ids.append(self.token_to_id.get(token, self.token_to_id.get("<|unk|>", 0)))
        
        return encoded_ids

    def decode(self, ids):
        tokens = [self.id_to_token.get(idx, "") for idx in ids]
        text = "".join(tokens).replace("</w>", " ")
        return text.strip()

# Demonstration
tokenizer = BPETokenizer(num_merges=30)
tokenizer.train(text)

sample = "text processing"
encoded = tokenizer.encode(sample)
decoded = tokenizer.decode(encoded)

print(f"Original: {sample}")
print(f"Encoded IDs: {encoded}")
print(f"Decoded: {decoded}")

### Conclusion

This implementation shows the fundamental mechanics of BPE. In production systems like OpenAI's `tiktoken`, the process is highly optimized using byte-level representations and more complex regex patterns (to handle whitespaces and punctuation better), but the core idea of iteratively merging frequent pairs remains the same.